# Forecast dataset inspection

Construction and admission audit for a forecast at **12:00 Europe/Brussels on D−1**. Build the versioned artifacts using the CLI before running this notebook. All joins, feature definitions, calendar calculations, and selectors live in `epgnn.dataset_build`.

Coverage checks inspect every split. Detailed test-target exploration is disabled by the package selector. This notebook fits no normalization or models. Conditional inputs are not certified for strict forecasting.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "epgnn").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from epgnn.dataset_build import inspection
CONFIG = ROOT / "configs/dataset_construction.json"

ZONES = ["BE", "FR", "DE_AT_LU", "DE_LU"]
START = "2024-03-25T00:00:00Z"
END = "2024-04-02T00:00:00Z"
VARIABLES = ["temp_pw", "load_fc", "n_presolved", "min_ram_ratio"]
meta = inspection.metadata(CONFIG)
display(pd.Series(meta["build_fingerprint"]))

## Missingness and structural gaps

The coverage table contains counts and classifications, including missing source rows, invalid values, structural lifetimes, and missing JAO domain publications. Nulls are retained. Missingness rates are coverage statistics, not test-target distributions.

In [ ]:
coverage = inspection.coverage(CONFIG, zones=ZONES)
numeric = coverage[coverage["missing"].notna()].copy()
numeric["missing_fraction"] = numeric["missing"] / numeric["rows"]
display(numeric.groupby(["zone", "split", "feature"], dropna=False)[["rows", "missing"]].sum().head(40))
display(coverage[coverage["classification"].notna()].groupby(["zone", "feature", "classification"], dropna=False)["rows"].sum().to_frame().head(40))
price_coverage = numeric[numeric.feature.eq("price")]
monthly = price_coverage.groupby(["zone", "calendar_month"])[["rows", "missing"]].sum().reset_index()
monthly["missing_fraction"] = monthly["missing"] / monthly["rows"]
heat = monthly.pivot(index="zone", columns="calendar_month", values="missing_fraction")
source_map = pd.DataFrame(meta["feature_registry"])[["feature", "table", "source"]]
by_source = numeric.merge(source_map, on=["feature", "table"], validate="many_to_one")
display(by_source.groupby(["table", "source", "split"])[["rows", "missing"]].sum().rename(columns={"rows": "feature_cells", "missing": "missing_feature_cells"}))
fig, ax = plt.subplots(figsize=(14, 3))
im = ax.imshow(heat.to_numpy(), aspect="auto", vmin=0, vmax=1, interpolation="nearest")
ax.set_yticks(range(len(heat.index)), heat.index)
step = max(1, len(heat.columns) // 10)
ax.set_xticks(range(0, len(heat.columns), step), heat.columns[::step], rotation=45)
ax.set_title("Price missingness by zone and UTC month, including structural absence")
fig.colorbar(im, ax=ax, label="fraction missing")
plt.tight_layout()
plt.show()

## Historical geography and DST

DE_AT_LU and DE_LU have distinct identities. The split is a local delivery-date boundary. Unresolved EIC mappings stay explicit. Germany-only weather is never substituted into the historical combined zone.

The timeline examples show winter/summer noon, 23/25-hour delivery days, repeated autumn hours, and origins on the day after each clock change.

In [ ]:
crosswalk = pd.DataFrame(meta["geography_crosswalk"])
display(crosswalk[crosswalk.zone.isin(["DE_AT_LU", "DE_LU", "AT"])])
dst = inspection.dst_examples()
display(dst.groupby("delivery_date").agg(hours=("ts_utc", "size"), origin=("forecast_origin_utc", "first")))
display(dst.loc[dst.delivery_date.isin(["2024-03-31", "2024-10-27"]), ["ts_utc", "delivery_date", "hour_local", "utc_offset_hours", "forecast_origin_utc"]])

## Selected source variables and lag reconstruction

Change the selectors above for a bounded date window. Raw features remain in physical units with NaNs. Observation, interval completion, and publication times are separate. Unknown publication times cannot yield strict eligibility.

In [ ]:
selected = inspection.select(CONFIG, ZONES, START, END, VARIABLES)
display(selected.head(20))
lag_cols = ["price_lag168", "price_lag168_observation_utc", "price_lag168_completion_utc", "price_lag168_publication_utc", "price_lag168_eligible"]
lag = inspection.select(CONFIG, ["BE"], START, END, lag_cols)
display(lag.head(12))
display(lag[["price_lag168_eligible"]].value_counts(dropna=False).to_frame("rows"))

## Strict, conditional, and excluded columns

The registry records source, transformation, units, geography, timing rule, admission, and unfitted normalization. By default archived price/weather/load/JAO values remain conditional. Strict historical price features require independently supported publication evidence and per-row gating. Same-day outcomes and source fill diagnostics are excluded.

In [ ]:
registry = pd.DataFrame(meta["feature_registry"])
display(registry.groupby(["table", "admission"]).size().to_frame("columns"))
display(registry[registry.admission.eq("admitted")])
examples = ["price_lag24", "price_lag48", "temp_pw_lag48", "load_fc", "n_presolved", "flow_phys", "maxbex", "price_source_PriceEUR_input"]
display(registry[registry.feature.isin(examples)])
strict = inspection.select(CONFIG, ["BE"], START, END, ["target_valid", "hour_local", "dow", "month", "in_core", "in_cwe"], table="strict")
display(strict.head())

## v1.0 differences and unresolved provenance

Comparison counts describe changes in alignment/missingness/values, without plotting target distributions. Differences are expected from the full source calendar, separate historical geography, exact lags, retained CWE features, and use of original unfilled values. Detailed CNECs stay in their original tables, referenced by content hash; cnec_key identifies observations rather than permanent assets.

In [ ]:
comparison = inspection.reference_comparison(CONFIG)
display(comparison.groupby(["table", "feature"])[["aligned_rows", "new_only_rows", "reference_only_rows", "differing_rows"]].sum())
display(pd.DataFrame(meta["findings"]))
display(meta["cnec_references"])